In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
pd.set_option('display.max_columns', 50)

In [3]:
# add your local pathname here

df = pd.read_csv('/Users/jamesemcnally/Dropbox/DSBC Student Risk Factors Datasets/merged_streaming_data.csv')
df.head()

,id_student,code_module,code_presentation,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,id_assessment,is_banked,score,assessment_type,due_date,weight,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,6516,AAA,2014J,-23.0,0,3,23,2,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
1,6516,AAA,2014J,-22.0,33,13,34,0,0,2,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
2,6516,AAA,2014J,-20.0,13,12,8,1,0,7,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
3,6516,AAA,2014J,-17.0,0,2,0,3,2,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
4,6516,AAA,2014J,-12.0,1,1,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN


In [4]:
df.columns

Index(['id_student', 'code_module', 'code_presentation', 'date', 'forumng',
       'homepage', 'oucontent', 'subpage', 'url', 'resource', 'glossary',
       'dataplus', 'oucollaborate', 'quiz', 'ouelluminate', 'sharedsubpage',
       'questionnaire', 'page', 'externalquiz', 'ouwiki', 'dualpane',
       'repeatactivity', 'folder', 'htmlactivity', 'id_assessment',
       'is_banked', 'score', 'assessment_type', 'due_date', 'weight',
       'module_presentation_length', 'gender', 'region', 'highest_education',
       'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits',
       'disability', 'final_result', 'date_registration',
       'date_unregistration'],
      dtype='object')

In [5]:
# Create a "week" column relative to each code_presentation's starting date. 
# Dates 0.0 - 6.0 are week 1, 7.0 - 13.0 are week 2, etc. 
# The weeks *before* date 0.0 are assigned to negative values.

df['week'] = df.groupby('code_presentation')['date'].transform(
    lambda x: ((x // 7) + 1).where(x >= 0, x // 7)
)

In [6]:
# Add submission related features for FIRST ASSIGNMENT only:
# - Relative submission date
# - Submission type (late vs. early)

# Find the first assignment for each student (by minimum due_date)
first_assessment = df[df['id_assessment'].notna()].groupby(['id_student', 'code_presentation'])['due_date'].min().reset_index()
first_assessment = first_assessment.rename(columns={'due_date': 'first_due_date'})

# Merge first assessment due date back to main dataframe
df = df.merge(first_assessment, on=['id_student', 'code_presentation'], how='left')

# For rows where the student submitted the first assessment (where due_date matches first_due_date and id_assessment exists)
# Calculate relative submission date and type
df_first_submission = df[(df['id_assessment'].notna()) & (df['due_date'] == df['first_due_date'])].copy()
df_first_submission['relative_submission_date'] = df_first_submission['first_due_date'] - df_first_submission['date']
df_first_submission['submission_type'] = np.where(df_first_submission['relative_submission_date'] < 0, "Late", "Early")

# Get one row per student with their first submission metrics
first_submission_metrics = df_first_submission.groupby(['id_student', 'code_presentation']).agg({
    'relative_submission_date': 'first',  # Take the submission date (should be one row per student for first assessment)
    'submission_type': 'first'
}).reset_index()

# Merge back to main dataframe
df = df.merge(first_submission_metrics, on=['id_student', 'code_presentation'], how='left', suffixes=('', '_first'))

# Clean up: if we created duplicate columns, keep the new ones
if 'relative_submission_date_first' in df.columns:
    df['relative_submission_date'] = df['relative_submission_date_first']
    df = df.drop(columns=['relative_submission_date_first'])
if 'submission_type_first' in df.columns:
    df['submission_type'] = df['submission_type_first']
    df = df.drop(columns=['submission_type_first'])

# Drop the helper column
df = df.drop(columns=['first_due_date'])

In [7]:
# Add feature that sums total VLE interactions through week 3

vle_columns = ['quiz', 'questionnaire', 'externalquiz', 'oucontent', 'page', 'resource', 'url', 'homepage', 
               'glossary', 'subpage', 'folder', 'forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage', 
               'dataplus', 'repeatactivity', 'dualpane', 'htmlactivity'
]

# Filter to pre-course through week 3 and sum interactions per student
df_pre_w3 = df[df['week'] <= 3]
total_interactions_pre_w3 = df_pre_w3.groupby(['id_student', 'code_presentation'])[vle_columns].sum().sum(axis=1)
total_interactions_pre_w3 = total_interactions_pre_w3.rename('total_vle_interactions_w3')
df = df.merge(total_interactions_pre_w3, on=['id_student', 'code_presentation'], how='left')

In [8]:
# Add VLE interaction category features:
# - Content type interaction (percentage)
# - Collaborative type interaction (percentage)

# This creates 2 columns for pre-course through week 3:
# - collaborative_focus_pre_w3
# - content_focus_pre_w3

content_types = ['oucontent', 'page', 'resource', 'url', 'homepage', 'glossary', 'subpage', 'folder']
collaborative_types = ['forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage']

# Pre-course through week 3
df_pre_w3 = df[df['week'] <= 3]
student_totals = df_pre_w3.groupby(['id_student', 'code_presentation'])[vle_columns].sum()

content_focus_pre_w3 = student_totals[content_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
content_focus_pre_w3 = content_focus_pre_w3.fillna(0).rename("content_focus_pre_w3")

collaborative_focus_pre_w3 = student_totals[collaborative_types].sum(axis=1) / student_totals[vle_columns].sum(axis=1)
collaborative_focus_pre_w3 = collaborative_focus_pre_w3.fillna(0).rename("collaborative_focus_pre_w3")

df = df.merge(content_focus_pre_w3, on=['id_student', 'code_presentation'], how='left')
df = df.merge(collaborative_focus_pre_w3, on=['id_student', 'code_presentation'], how='left')

In [9]:
# Add regularity features for pre-course through week 3:
# - Active days per week
# - Standard deviation of gaps between logins

# This creates 2 columns:
# - active_days_per_week_pre_w3
# - std_regularity_pre_w3

def calculate_weekly_consistency(df_filtered, suffix, student_id_col='id_student', date_col='date', week_col='week'):
    weekly_days = df_filtered.groupby([student_id_col, week_col])[date_col].nunique()
    consistency = weekly_days.groupby(student_id_col).std()
    consistency = consistency.rename(f'active_days_per_week_{suffix}')
    return consistency

def calculate_regularity_std_period(df_filtered, suffix, student_id_col='id_student', date_col='date'):
    regularity = df_filtered.groupby(student_id_col)[date_col].apply(
        lambda x: x.sort_values().diff().std()
    )
    regularity = regularity.rename(f'std_regularity_{suffix}')
    return regularity

# Pre-course through week 3
df_pre_w3 = df[df['week'] <= 3]
active_days_pre_w3 = calculate_weekly_consistency(df_pre_w3, 'pre_w3')
std_regularity_pre_w3 = calculate_regularity_std_period(df_pre_w3, 'pre_w3')

# Merge both metrics back to original dataframe
df['active_days_per_week_pre_w3'] = df['id_student'].map(active_days_pre_w3)
df['std_regularity_pre_w3'] = df['id_student'].map(std_regularity_pre_w3)

In [10]:
# Add diversity of interaction features for pre-course through week 3
# - VLE richness (number of different VLE types used) 
# - Shannon entropy (overall diversity of interactions)

# This creates 2 columns:
# - vle_richness_pre_w3
# - diversity_shannon_pre_w3

from scipy.stats import entropy

def shannon_entropy_calc(counts, vle_columns):
    counts = counts[vle_columns].values.astype(float)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return 0
    proportions = counts / counts.sum()
    return entropy(proportions, base=2)

def calculate_vle_metrics(df_filtered, suffix, vle_columns):
    # Group by student and sum VLE activities for the period
    student_vle = df_filtered.groupby('id_student')[vle_columns].sum()
    
    # Calculate richness (number of VLE types used)
    richness = (student_vle > 0).sum(axis=1).rename(f'vle_richness_{suffix}')
    
    # Calculate Shannon entropy
    diversity = student_vle.apply(lambda row: shannon_entropy_calc(row, vle_columns), axis=1).rename(f'diversity_shannon_{suffix}')
    
    return richness, diversity

# Pre-course through week 3
df_pre_w3 = df[df['week'] <= 3]
vle_richness_pre_w3, diversity_shannon_pre_w3 = calculate_vle_metrics(df_pre_w3, 'pre_w3', vle_columns)

# Merge both metrics back to original dataframe
df['vle_richness_pre_w3'] = df['id_student'].map(vle_richness_pre_w3)
df['diversity_shannon_pre_w3'] = df['id_student'].map(diversity_shannon_pre_w3)

In [11]:
df.columns

Index(['id_student', 'code_module', 'code_presentation', 'date', 'forumng',
       'homepage', 'oucontent', 'subpage', 'url', 'resource', 'glossary',
       'dataplus', 'oucollaborate', 'quiz', 'ouelluminate', 'sharedsubpage',
       'questionnaire', 'page', 'externalquiz', 'ouwiki', 'dualpane',
       'repeatactivity', 'folder', 'htmlactivity', 'id_assessment',
       'is_banked', 'score', 'assessment_type', 'due_date', 'weight',
       'module_presentation_length', 'gender', 'region', 'highest_education',
       'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits',
       'disability', 'final_result', 'date_registration',
       'date_unregistration', 'week', 'relative_submission_date',
       'submission_type', 'total_vle_interactions_w3', 'content_focus_pre_w3',
       'collaborative_focus_pre_w3', 'active_days_per_week_pre_w3',
       'std_regularity_pre_w3', 'vle_richness_pre_w3',
       'diversity_shannon_pre_w3'],
      dtype='object')

In [12]:
# Dropping rows in which a student already took the course

df = df[df['num_of_prev_attempts'] != 1]

In [21]:
# Drop students who withdrew before week 4 (i.e., withdrew during weeks 1-3 or pre-course)

# Filter to remove students who withdrew before week 4
df = df[~((df['final_result'] == 'Withdrawn') & (df['date_unregistration'] <= 20))]

print(f"Remaining students after filtering: {df['id_student'].nunique()}")
print(f"Remaining rows: {len(df)}")
print(f"\nFinal result distribution:")
print(df.groupby('final_result')['id_student'].nunique())

Remaining students after filtering: 22829
Remaining rows: 1710577

Final result distribution:
final_result
Distinction     2764
Fail            5596
Pass           10926
Withdrawn       4476
Name: id_student, dtype: int64


In [13]:
# Collapse dataframe to one row per student
# Keep only student-level features (demographics and aggregated metrics)

# Define columns to keep (student-level features that don't vary by date)
# Note: id_student and code_presentation will be the grouping keys
student_level_columns = [
    # Demographics
    'code_module',
    'gender', 
    'region', 
    'highest_education', 
    'imd_band', 
    'age_band', 
    'num_of_prev_attempts',
    'studied_credits', 
    'disability', 
    'final_result',
    'date_registration',
    'date_unregistration',
    'module_presentation_length',
    # Aggregated features
    'total_vle_interactions_w3',
    'content_focus_pre_w3',
    'collaborative_focus_pre_w3',
    'active_days_per_week_pre_w3',
    'std_regularity_pre_w3',
    'vle_richness_pre_w3',
    'diversity_shannon_pre_w3',
    # First assessment metrics
    'relative_submission_date',
    'submission_type'
]

# Check which columns exist in the dataframe
available_columns = [col for col in student_level_columns if col in df.columns]

# Group by student and presentation, taking the first value for each column
# (since student-level features should be the same across all rows for a given student)
df_student_level = df.groupby(['id_student', 'code_presentation'])[available_columns].first().reset_index()

print(f"Original dataframe shape: {df.shape}")
print(f"Student-level dataframe shape: {df_student_level.shape}")
print(f"\nColumns in student-level dataframe: {len(df_student_level.columns)}")
print(f"\nFirst few rows:")
df_student_level.head()

Original dataframe shape: (1722342, 52)
Student-level dataframe shape: (25559, 24)

Columns in student-level dataframe: 24

First few rows:


,id_student,code_presentation,code_module,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,total_vle_interactions_w3,content_focus_pre_w3,collaborative_focus_pre_w3,active_days_per_week_pre_w3,std_regularity_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3,relative_submission_date,submission_type
0,6516,2014J,AAA,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,606.0,0.739274,0.260726,1.812654,1.358621,6.0,2.094273,2.0,Early
1,8462,2013J,DDD,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0,261,317.0,0.886435,0.107256,1.414214,0.825578,9.0,2.357050,-4.0,Late
2,11391,2013J,AAA,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,268,401.0,0.862843,0.137157,1.414214,2.627691,6.0,1.479023,1.0,Early
3,23629,2013B,BBB,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN,240,51.0,0.372549,0.627451,0.500000,2.872281,4.0,1.424794,10.0,Early
4,23698,2014J,CCC,F,East Anglian Region,A Level or Equivalent,50-60%,0-35,0,120,N,Pass,-110.0,NaN,269,352.0,0.241477,0.068182,2.160247,0.770281,7.0,1.503976,-3.0,Late


Separate modules and save to csv

In [23]:
# Create a folder to store the CSV files (optional but recommended)
output_folder = "modules_csv"
os.makedirs(output_folder, exist_ok=True)

# Filter to only include modules AAA, CCC, DDD, and FFF
selected_modules = ['AAA', 'CCC', 'DDD', 'FFF']
df_filtered = df[df['code_module'].isin(selected_modules)]

# Save all selected modules together in a single file
filename = f"{output_folder}/combined_modules_data.csv"
df_filtered.to_csv(filename, index=False)
print(f"Saved: {filename}")
print(f"Included modules: {', '.join(selected_modules)}")
print(f"Total rows: {len(df_filtered)}")

Saved: modules_csv/combined_modules_data.csv
Included modules: AAA, CCC, DDD, FFF
Total rows: 1146352
